In [3]:
!pip install -U "transformers[torch]" sentencepiece

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-es-en"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Using:", device)

config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Using: cuda


In [5]:
sentences = [
    "Hola, ¿cómo estás?",
    "Me gusta mucho aprender programación."
]

inputs = tokenizer(
    sentences,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=50
    )

translations = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)

for spanish, english in zip(sentences, translations):
    print("Spanish :", spanish)
    print("English :", english)
    print()

Spanish : Hola, ¿cómo estás?
English : Hey, how are you?

Spanish : Me gusta mucho aprender programación.
English : I really like learning programming.



In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device: cuda


In [7]:
import torch.nn.functional as F

sentence = "The weather today is"

inputs = tokenizer(sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :]

# Convert logits → probabilities
probabilities = F.softmax(logits, dim=-1)

# Get top 5
top_probs, top_ids = torch.topk(probabilities, 5)

for prob, token_id in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode(token_id)
    print(f"{repr(token):15} {prob.item():.4f}")

' very'         0.0428
' good'         0.0322
' pretty'       0.0306
' a'            0.0300
' not'          0.0253


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Using:", device)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Using: cuda


In [9]:
sentences = [
    "I absolutely loved this movie!",
    "This is the worst product I have ever bought.",
    "The food was amazing.",
    "I am very disappointed with the service.",
    "The movie was okay."
]

inputs = tokenizer(
    sentences,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)
predictions = torch.argmax(probabilities, dim=-1)

labels = ["NEGATIVE", "POSITIVE"]

for sentence, prediction, probability in zip(
    sentences,
    predictions,
    probabilities
):
    print(
        f"{sentence}\n"
        f"→ {labels[prediction.item()]} "
        f"({probability[prediction].item():.4f})\n"
    )

I absolutely loved this movie!
→ POSITIVE (0.9999)

This is the worst product I have ever bought.
→ NEGATIVE (0.9998)

The food was amazing.
→ POSITIVE (0.9999)

I am very disappointed with the service.
→ NEGATIVE (0.9998)

The movie was okay.
→ POSITIVE (0.9998)

